In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/flight-delays-fall-2018/sample_submission.csv.zip
/kaggle/input/competitions/flight-delays-fall-2018/flight_delays_train.csv.zip
/kaggle/input/competitions/flight-delays-fall-2018/flight_delays_test.csv.zip


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Данные соревнования доступны только для чтения в /kaggle/input
print(os.listdir('/kaggle/input'))

['competitions']


In [3]:
DATA = '/kaggle/input/flight-delays-fall-2018'
# Читаем обучающую и тестовую выборку
train = pd.read_csv('/kaggle/input/competitions/flight-delays-fall-2018/flight_delays_train.csv.zip')
test = pd.read_csv('/kaggle/input/competitions/flight-delays-fall-2018/flight_delays_test.csv.zip')
# Выводим первые 5 строчек таблицы
print('train:', train.shape, '| test:', test.shape)
train.head()

train: (100000, 9) | test: (100000, 8)


,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y


Данные успешно загружены и разделены на обучающую и тестовую выборки. Обучающий набор содержит как числовые, так и категориальные признаки, а также бинарную целевую переменную dep_delayed_15min, в то время как на тестовом наборе нам предстоит спрогнозировать факт задержки рейса. Разметка целевого класса представлена строковыми значениями «Y» и «N», что потребует их предварительного перевода в числовой формат (1 и 0) перед обучением моделей.

In [4]:
# 1. Проверяем общую размерность таблицы (количество строк и столбцов)
print("Размер обучающей выборки:", train.shape)

# 2. Оцениваем распределение целевой переменной (баланс классов 'N' и 'Y')
print("\nРаспределение ответов:")
print(train['dep_delayed_15min'].value_counts())

# 3. Выводим техническую сводку: типы данных, количество заполненных строк и расход памяти
print("\nИнформация о столбцах:")
train.info()

Размер обучающей выборки: (100000, 9)

Распределение ответов:
dep_delayed_15min
N    80956
Y    19044
Name: count, dtype: int64

Информация о столбцах:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Month              100000 non-null  object
 1   DayofMonth         100000 non-null  object
 2   DayOfWeek          100000 non-null  object
 3   DepTime            100000 non-null  int64 
 4   UniqueCarrier      100000 non-null  object
 5   Origin             100000 non-null  object
 6   Dest               100000 non-null  object
 7   Distance           100000 non-null  int64 
 8   dep_delayed_15min  100000 non-null  object
dtypes: int64(2), object(7)
memory usage: 6.9+ MB


Набор данных состоит из 100 000 строк и 9 признаков, при этом пропущенные значения (NaN) полностью отсутствуют. В данных наблюдается заметный дисбаланс классов: рейсы без задержки (N) составляют примерно 81% выборки, тогда как задержанные рейсы (Y) — около 19%. Полученный дисбаланс критически важно учесть при выборе стратегии валидации (StratifiedKFold) и метрики оценки качества, чтобы модель не уходила в банальное предсказание мажоритарного класса.